# Week 5, Day 1: Brute-force RAG for WWE questions

**Lesson:** [week5/day1.ipynb](../../../week5/day1.ipynb), *RAG week begins: an expert knowledge worker*. **About 35 minutes.**

**What you practice here**

- Brute-force retrieval: looking up each word of the question in a dictionary of documents, as the lesson's `get_relevant_context` does.
- Adding what was found to the system prompt, with a sentence of your own when nothing was found, as the lesson's `SYSTEM_PREFIX` and `additional_context` do.
- The RAG chat function that Gradio calls: a system message with the context, then the history, then the user's message, sent with `client.chat.completions.create`, and the reply's text read from the response.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week5/day1", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson's assistant knows Insurellm's employees by their last names. It reads each file into a dictionary under
that name, looks for the names in the question, and pastes the files it finds into the system prompt. Build the same
assistant for your WWE documents, where a wrestler's file is found by one word of the ring name: `hogan` for
`people/hulk-hogan.md`, `undertaker` for `people/the-undertaker.md`.

The parts that look things up take that dictionary, `knowledge`, as a parameter. The one that calls the model,
`chat`, takes the client as its first parameter and makes the Chat Completions call itself, as the lesson's `chat`
does. The checks hand it a stand-in shaped like the real client, which records what you send and answers with a
canned reply, so nothing is spent. In Try it for real you build the dictionary from your files and give `chat` to
Gradio.

## What each call should give

```python
knowledge = {"hogan": "# Hulk Hogan\n...", "giant": "# André the Giant\n...", "edge": "# Edge\n..."}

relevant_docs("Did Hogan ever beat André the Giant?", knowledge)
# ['# Hulk Hogan\n...', '# André the Giant\n...']

relevant_docs("What was the Undertaker's streak?", {"undertaker": "# The Undertaker\n..."})
# ['# The Undertaker\n...']

relevant_docs("What is in the knowledge base?", knowledge)
# []    ("edge" inside "knowledge" is not a word of the question)

additional_context("Who is Edge?", knowledge)
# 'The following context might help answer the question:\n\n# Edge\n...'    (the opening line is yours)

chat(client, "When did Edge enter the Hall of Fame?", history, knowledge)
# makes one request with the model gpt-4.1-nano and these messages: a system message holding SYSTEM_PREFIX and
# the context for Edge, then the history, then the question as a user message; and returns the reply's text
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `relevant_docs(question, knowledge)`

Finds the documents whose lookup word is a word of the question. `knowledge` maps a lowercase word to a document's
text, such as `{"hogan": "# Hulk Hogan\n...", "undertaker": "# The Undertaker\n..."}`. Lowercase the question, turn
every character that is not a letter into a space, and split what is left into words. So "Undertaker's" gives the
words `undertaker` and `s`, where the lesson's cleaning, which drops the apostrophe, would give `undertakers` and miss.

Return a list of the texts whose key is one of those words, in the order the words come in the question. Give each
text at most once: a name asked twice, or two keys for the same text, still gives it once. Return an empty list when no
word is a key.

<details><summary>Hint</summary>

A string's `isalpha()` tells a letter from anything else, and `split()` cuts at the spaces:

```python
text = "".join(ch if ch.isalpha() else " " for ch in question.lower())
for word in text.split():
    ...
```

</details>

In [ ]:
def relevant_docs(question, knowledge):
    # Write relevant_docs here, as described above.
    raise NotImplementedError("relevant_docs is not written yet")

In [ ]:
check("relevant_docs")

### 2 · `additional_context(question, knowledge)`

Builds the context to add to the system prompt, from what `relevant_docs(question, knowledge)` finds. When it finds
nothing, return a sentence of your own saying there is no additional context for this question. Otherwise return an
opening line of your own that introduces the context, then the documents in the order found, with a blank line
between documents, as the lesson's `additional_context` does.

<details><summary>Hint</summary>

```python
return "Your opening line:\n\n" + "\n\n".join(docs)
```

</details>

In [ ]:
def additional_context(question, knowledge):
    # Write additional_context here, as described above.
    raise NotImplementedError("additional_context is not written yet")

In [ ]:
check("additional_context")

### 3 · `SYSTEM_PREFIX`

The start of your system prompt: `chat` adds the context straight after it. Like the lesson's, it says who the
assistant is (an expert on the history of WWE), asks for brief, accurate answers, and says what to do when it does
not know the answer. The check looks only for "WWE" and the word "know"; the rest of the wording is yours.

<details><summary>Hint</summary>

A triple-quoted string, as the lesson's is, ending with a line that introduces the context:

```python
SYSTEM_PREFIX = """You are ... (who the assistant is)
... (how to answer, and what to do when it does not know)

Relevant context:
"""
```

</details>

In [ ]:
SYSTEM_PREFIX = ""

In [ ]:
check("SYSTEM_PREFIX")

### 4 · `chat(client, message, history, knowledge, model="gpt-4.1-nano")`

Answers one message with the context its words point to, in one Chat Completions call, and returns the reply's text.
`history` is what Gradio's chat interface passes in its `"messages"` mode: a list of messages, each a dict with a
`"role"` and a `"content"`.

Make one call with the model `model` and these messages: a system message whose content is `SYSTEM_PREFIX` followed
by `additional_context(message, knowledge)`, then the history messages as they are, then a user message holding
`message`. Look up context for the new message only, not the history. Return the reply's text as it is. Build a new
list of messages: the `history` you were given must not change.

<details><summary>Hint</summary>

```python
system = SYSTEM_PREFIX + additional_context(message, knowledge)
messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
response = client.chat.completions.create(model=model, messages=messages)
return response.choices[0].message.content
```

</details>

In [ ]:
def chat(client, message, history, knowledge, model="gpt-4.1-nano"):
    # Write chat here, as described above.
    raise NotImplementedError("chat is not written yet")

In [ ]:
check("chat")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 5, Day 1 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Each answer from `gpt-4.1-nano` costs a fraction of a cent, even with a few documents in its prompt.

1. Load your keys with `load_dotenv(override=True)` and make `client = OpenAI()`.
2. Build `knowledge` from your people files, `ROOT / "plan" / "docs" / "people"` (the setup cell set `ROOT` to your
   fork's folder). The key is the last word of the file name without `.md`, split at `-` (`hulk-hogan.md` gives
   `hogan`); the value is the file's text, read as UTF-8, as the lesson does with last names.
3. Print `additional_context` for "How did the Undertaker's WrestleMania streak end?", then `chat`'s answer to it.
   Then ask something that names nobody in `knowledge`, such as "Who main-evented WrestleMania III?", and compare.
4. Launch the chat: `gr.ChatInterface` with a function of `message` and `history` that calls your `chat`, and
   `type="messages"`, then `.launch(inbrowser=True)`.

<details><summary>Stuck? One way to write it</summary>

```python
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

knowledge = {}
for path in (ROOT / "plan" / "docs" / "people").glob("*.md"):
    knowledge[path.stem.split("-")[-1].lower()] = path.read_text(encoding="utf-8")
print(len(knowledge), "people:", sorted(knowledge)[:8])

question = "How did the Undertaker's WrestleMania streak end?"
print(additional_context(question, knowledge)[:300])
print(chat(client, question, [], knowledge))
print(chat(client, "Who main-evented WrestleMania III?", [], knowledge))  # names nobody, so no context

gr.ChatInterface(lambda message, history: chat(client, message, history, knowledge),
                 type="messages").launch(inbrowser=True)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.